# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset covers ordered logistic regression outputs on predictors for indigenous and modern knowledge adoption in rangeland management by pastoral households in Northern Kenya.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's inspect available record sets, their `@id`s, and all fields for further exploration.

In [ ]:
# Get all record sets defined in the dataset schema
record_sets = dataset.record_sets

print("### Record Sets (@id, name):")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")

# Show all fields in each record set
for rs in record_sets:
    print(f"\n-- Record Set: {rs.get('name', '[no name]')} (@id: {rs['@id']}) --")
    if 'field' in rs:
        for field in rs['field']:
            if isinstance(field, dict):
                print(f"  - Field @id: {field['@id']}, name: {field.get('name', '[no name]')}, dataType: {field.get('dataType', '[not specified]')}")
            else:
                print(f"  - Field: {field}")
    else:
        print("  [No fields declared]")

## 3. Data Extraction

Load data from available record sets into Pandas DataFrames. Reference each record set and field by their `@id` for clarity and reproducibility.

In [ ]:
# List of record set @id's to extract data from
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

print("Available record sets in dataset:")
for idx, rs_id in enumerate(record_set_ids):
    print(f"  {idx}: {rs_id}")

# Extract records for each recordset into a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Demonstrate columns in the first non-empty DataFrame
for rs_id, df in dataframes.items():
    print(f"\nRecord Set @id: {rs_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())
    # We'll use the first non-empty dataframe for EDA
    main_record_set_id = rs_id
    break
else:
    print("No records found in any record set.")


## 4. Exploratory Data Analysis (EDA)

Let's process the data. We'll select a numeric field and a group/categorical field for filtering, normalization, and aggregation.

_**Note:** Modify `numeric_field_id` and `group_field_id` variables below based on actual fields in your dataset (as found in Section 2/3 above)._

In [ ]:
# --- Configure these based on your dataset schema ---
# Use actual @id's from the chosen record set (see Section 2/3 output)
# For demonstration, we'll automatically pick first numeric and group fields if available
df = dataframes[main_record_set_id]

import numpy as np

# Identify likely numeric and group field candidates
numeric_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int] and not col.startswith('Unnamed')]
if not numeric_candidates:
    # Try to infer numeric columns by converting
    for col in df.columns:
        try:
            df[col+'_conv'] = pd.to_numeric(df[col], errors='coerce')
            if df[col+'_conv'].notnull().sum() > 0:
                numeric_candidates.append(col)
        except Exception:
            continue

# Restore original column if conversion happened
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    # Use 'df[numeric_field_id]' as float; make sure it's numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
else:
    print("No numeric field found for EDA.")
    numeric_field_id = None

# Use a group/categorical field if available (other than the numeric one)
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and (df[col].dtype == object or df[col].dtype.name == "category"):
        group_field_id = col
        break

if numeric_field_id:
    # Filter: e.g., values above the mean
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group and aggregate if group_field_id is found
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nAverage {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field detected for EDA. Please check your record set fields.")

## 5. Visualization

Plot distributions and relationships between fields (where possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group, if grouping exists
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} distribution by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field to plot. Visualization skipped.")

## 6. Conclusion

- This notebook demonstrated loading and exploring a FAIR² dataset using the Croissant schema and `mlcroissant` library.
- We highlighted the use of `@id` for all entity references, and dynamically explored, filtered, and visualized the data.
- For rigorous future analysis, consult the Croissant schema field definitions to select the most relevant predictors and outcomes for your research question.